<a href="https://colab.research.google.com/github/ryo-67/body-politic/blob/main/VidScraper_Updated_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Video Scraper

Before anything, upload
1. video_links.xlsx
2. failed_transcripts.xlsx

If you have uploaded a failed_transcripts.xlsx file, it will take only the URLS there.

In either case, a new failed_transcripts.xlsx will be generated

## Dependencies

In [ ]:
import sys

!pip install --quiet youtube-transcript-api yt-dlp pandas dateparser

!pip install yt-dlp pandas

!pip install dateparser
import dateparser
from datetime import datetime

!apt-get update -y
!apt-get install -y nodejs npm
!node -v

!pip install --upgrade youtube-transcript-api
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound


import yt_dlp
import pandas as pd
import json
import time
import requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 20.8 MB/s eta 0:00:00
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [90.8 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://ar

## Test Extraction - Metadata

In [ ]:
# Excel file should be named 'video_links.xlsx' (upload to directory),
# and add a column with videos named 'video_url'.
try:
    video_links_df = pd.read_excel('video_links.xlsx')
    video_urls = video_links_df['video_url'].tolist()
except FileNotFoundError:
    print("Error: 'video_links.xlsx' not found. Please make sure the file is in the correct directory.")
    # Fallback to the original single URL for demonstration if the file is not found
    video_urls = ["https://www.youtube.com/watch?v=dQw4w9WgXcQ"]
except KeyError:
    print("Error: 'video_url' column not found in 'video_links.xlsx'. Please ensure the column name is correct.")
    video_urls = ["https://www.youtube.com/watch?v=dQw4w9WgXcQ"]

def extract_metadata(url):
    ydl_opts = {
    "quiet": True,
    "skip_download": True,
    "extract_flat": False,
    "extractor_args": {
        "youtube": {
            "getcomments": True,
            "maxcomments": 200
        }
    },
    "js_runtimes": {"node": {"path": "/usr/bin/node"}},
    "no_warnings": True
}

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)

    return info

def normalize_metadata(info):
    data = {
        "video_id": info.get("id"),
        "title": info.get("title"),
        "description": info.get("description"),
        "channel": info.get("uploader"),
        "channel_id": info.get("channel_id"),
        "upload_date": info.get("upload_date"),
        "duration_seconds": info.get("duration"),
        "view_count": info.get("view_count"),
        "like_count": info.get("like_count"),
        "comment_count": info.get("comment_count"),
        "tags": info.get("tags"),
        "categories": info.get("categories"),
        "webpage_url": info.get("webpage_url"),
        "language": info.get("language"),
        "availability": info.get("availability")
    }

    return pd.DataFrame([data])

all_metadata_dfs = []
for url in video_urls:
    print(f"Extracting metadata for: {url}")
    try:
        info = extract_metadata(url)
        if info: # Ensure info is not empty/None
            df_row = normalize_metadata(info)
            all_metadata_dfs.append(df_row)
        else:
            print(f"No metadata extracted for {url}")
    except Exception as e:
        print(f"Failed to extract metadata for {url}: {e}")

if all_metadata_dfs:
    metadata_df = pd.concat(all_metadata_dfs, ignore_index=True)
else:
    print("No metadata could be extracted for any of the provided URLs.")
    metadata_df = pd.DataFrame(columns=[
        "video_id", "title", "description", "channel", "channel_id",
        "upload_date", "duration_seconds", "view_count", "like_count",
        "comment_count", "tags", "categories", "webpage_url", "language",
        "availability"
    ]) # Create an empty DataFrame with expected columns

metadata_df["upload_date"] = pd.to_datetime(
    metadata_df["upload_date"], format="%Y%m%d", errors="coerce"
)

cols = ["view_count", "like_count", "comment_count", "duration_seconds"]
# Apply to_numeric only if the columns exist in the DataFrame
existing_cols = [col for col in cols if col in metadata_df.columns]
if existing_cols:
    metadata_df[existing_cols] = metadata_df[existing_cols].apply(pd.to_numeric, errors="coerce")

print("\n--- Consolidated Metadata DataFrame (first 5 rows) ---")
print(metadata_df.head())

Error: 'video_links.xlsx' not found. Please make sure the file is in the correct directory.
Extracting metadata for: https://www.youtube.com/watch?v=dQw4w9WgXcQ

--- Consolidated Metadata DataFrame (first 5 rows) ---
      video_id                                              title  \
0  dQw4w9WgXcQ  Rick Astley - Never Gonna Give You Up (Officia...   

                                         description      channel  \
0  The official video for “Never Gonna Give You U...  Rick Astley   

                 channel_id upload_date  duration_seconds  view_count  \
0  UCuAXFkgsw1L7xaCfnd5JJOw  2009-10-25               213  1769370874   

   like_count  comment_count  \
0    19075291        2400000   

                                                tags categories  \
0  [rick astley, Never Gonna Give You Up, nggyu, ...    [Music]   

                                   webpage_url language availability  
0  https://www.youtube.com/watch?v=dQw4w9WgXcQ       en       public  


Resulting Dataframe:

In [ ]:
metadata_df

,video_id,title,description,channel,channel_id,upload_date,duration_seconds,view_count,like_count,comment_count,tags,categories,webpage_url,language,availability
0,dQw4w9WgXcQ,Rick Astley - Never Gonna Give You Up (Officia...,The official video for “Never Gonna Give You U...,Rick Astley,UCuAXFkgsw1L7xaCfnd5JJOw,2009-10-25,213,1769370874,19075291,2400000,"[rick astley, Never Gonna Give You Up, nggyu, ...",[Music],https://www.youtube.com/watch?v=dQw4w9WgXcQ,en,public


## Test Extraction - Transcripts

## Functions

In [ ]:
def get_transcript_auto(video_id, languages=["en"], http_client=None):
    try:
        if http_client:
            ytt_api = YouTubeTranscriptApi(http_client=http_client)
        else:
            ytt_api = YouTubeTranscriptApi()

        transcript_list = ytt_api.list(video_id)
        transcript = None

        # Combine preferred languages with common English variants for a robust search
        # Use a set to avoid duplicates and preserve order for preferred languages first
        all_english_variants = list(dict.fromkeys(languages + ["en", "en-US", "en-GB"]))

        # 1. Try to find auto-generated transcripts with all English variants
        try:
            transcript = transcript_list.find_generated_transcript(all_english_variants)
        except NoTranscriptFound:
            pass  # If not found, proceed to manually uploaded

        # 2. If no auto-generated found, try to find manually uploaded transcripts with all English variants
        if transcript is None:
            try:
                transcript = transcript_list.find_transcript(all_english_variants)
            except NoTranscriptFound:
                pass  # If still not found, raise an error later

        # If after all attempts, no transcript is found, return empty DF and an error message
        if transcript is None:
            return pd.DataFrame(columns=["text", "start", "duration", "video_id"]), f"No suitable English transcript found for video ID {video_id} using languages {all_english_variants}."

        raw_data = transcript.fetch().to_raw_data()
        df = pd.DataFrame(raw_data)
        df["video_id"] = video_id
        return df, None # Return df and no error message (None) for success

    except TranscriptsDisabled as e:
        return pd.DataFrame(columns=["text", "start", "duration", "video_id"]), f"Transcripts Disabled: {e}"
    except NoTranscriptFound as e:
        return pd.DataFrame(columns=["text", "start", "duration", "video_id"]), f"No Transcript Found: {e}"
    except Exception as e:
        return pd.DataFrame(columns=["text", "start", "duration", "video_id"]), f"An unexpected error occurred within get_transcript_auto for {video_id}: {e}"

Dataframe:

In [ ]:
import requests # Import requests
import os
# from youtube_transcript_api.exceptions import RequestBlocked, IPBlocked # Removed due to ModuleNotFoundError
from requests.exceptions import SSLError
import time # Import the time module
import random # Import the random module for shuffling

# List of proxy dictionaries. Replace with your actual proxies.
# Example:
# PROXY_LIST = [
#     {'http': 'http://YOUR_PROXY_IP_1:YOUR_PROXY_PORT_1', 'https': 'http://YOUR_PROXY_IP_1:YOUR_PROXY_PORT_1'},
#     {'http': 'http://YOUR_PROXY_IP_2:YOUR_PROXY_IP_2', 'https': 'http://YOUR_PROXY_IP_2:YOUR_PROXY_IP_2'},
# ]
# Default placeholder if no real proxies are provided
DEFAULT_PROXY = {'http': 'http://YOUR_PROXY_IP:YOUR_PROXY_PORT', 'https': 'http://YOUR_PROXY_IP:YOUR_PROXY_PORT'}
PROXY_LIST = [
    {'http': 'http://YOUR_PROXY_IP:YOUR_PROXY_PORT', 'https': 'http://YOUR_PROXY_IP:YOUR_PROXY_PORT'},
    # Add more proxies here if you have them, or replace with your own reliable proxies
]

# Delay between each video's extraction attempts to avoid blocking
DELAY_SECONDS = 2

session = requests.Session() # Session is not directly used for proxying youtube-transcript-api, but kept for other requests if needed.

# Check if real proxies are configured
if PROXY_LIST and PROXY_LIST[0] != DEFAULT_PROXY:
    print(f"Using initial proxy: {PROXY_LIST[0]['http']}")
else:
    print("No real proxies configured. Using direct connection. IP blocking might occur.")


all_transcripts_dfs = []
failed_video_details = [] # New list to store detailed failure info

# Get all unique video IDs from the metadata_df (which is populated from video_links.xlsx)
all_video_ids_from_metadata = list(metadata_df["video_id"].unique())

video_ids_to_process = []
failed_transcripts_filename = "failed_transcripts.xlsx"

if os.path.exists(failed_transcripts_filename):
    try:
        failed_df_existing = pd.read_excel(failed_transcripts_filename)
        if not failed_df_existing.empty and "video_id" in failed_df_existing.columns:
            video_ids_to_process = list(failed_df_existing["video_id"].unique())
            print(f"'{failed_transcripts_filename}' found. Processing only {len(video_ids_to_process)} video IDs from this file.")
        else:
            print(f"'{failed_transcripts_filename}' found but is empty or missing 'video_id' column. Falling back to all videos from 'video_links.xlsx'.")
            video_ids_to_process = all_video_ids_from_metadata
    except Exception as e:
        print(f"Error reading '{failed_transcripts_filename}' for exclusive processing: {e}. Falling back to all videos from 'video_links.xlsx'.")
        video_ids_to_process = all_video_ids_from_metadata
else:
    print(f"No '{failed_transcripts_filename}' found. Processing all videos from 'video_links.xlsx'.")
    video_ids_to_process = all_video_ids_from_metadata

random.shuffle(video_ids_to_process)

print(f"Total video IDs to process: {len(video_ids_to_process)}")

for video_id in video_ids_to_process:
    print(f"Extracting transcript for video ID: {video_id}")
    attempts = 0
    transcript_extracted = False
    current_proxy_index = 0
    max_proxy_attempts_per_video = len(PROXY_LIST) if PROXY_LIST and PROXY_LIST[0] != DEFAULT_PROXY else 1
    current_failure_reason = None # To store the specific reason for the current video

    while attempts < max_proxy_attempts_per_video and not transcript_extracted:
        try:
            # Set environment variables for the current proxy for youtube-transcript-api to pick up
            if PROXY_LIST and PROXY_LIST[0] != DEFAULT_PROXY:
                proxy = PROXY_LIST[current_proxy_index % len(PROXY_LIST)]
                os.environ['HTTP_PROXY'] = proxy['http']
                os.environ['HTTPS_PROXY'] = proxy['https']
                print(f"  Attempting with proxy: {os.environ['HTTPS_PROXY']}")
            else:
                # Ensure no proxy environment variables are set for direct connection
                if 'HTTP_PROXY' in os.environ:
                    del os.environ['HTTP_PROXY']
                if 'HTTPS_PROXY' in os.environ:
                    del os.environ['HTTPS_PROXY']
                print("  Attempting with direct connection (no proxy configured).")

            transcript_for_video, error_message = get_transcript_auto(video_id)

            if error_message:
                print(f"  Failure during transcript extraction for {video_id}: {error_message}")
                current_failure_reason = error_message
                # If the error is about transcript being disabled or not found, it's a permanent failure for this video
                if "Transcripts Disabled" in error_message or "No suitable English transcript found" in error_message:
                    transcript_extracted = True # Mark as done (failed) for this video_id
                    print(f"  No further attempts for {video_id} due to non-proxy related transcript issue.")
                else:
                    # For other errors from get_transcript_auto (e.g., unexpected internal), retry with next proxy
                    attempts += 1
                    if attempts < max_proxy_attempts_per_video:
                        current_proxy_index += 1
                        print(f"  Retrying {video_id} with next proxy for internal error.")
                    else:
                        print(f"  All {max_proxy_attempts_per_video} proxies exhausted for {video_id} for internal error. Skipping.")
            else: # Transcript successfully extracted (error_message is None)
                all_transcripts_dfs.append(transcript_for_video)
                transcript_extracted = True
                print(f"  Successfully extracted transcript for {video_id}.")

        except (Exception, SSLError) as e: # This catches network/proxy related issues
            current_failure_reason = f"IP Blocking/Proxy Error: {e}"
            print(f"  {current_failure_reason} for {video_id} with proxy {os.environ.get('HTTPS_PROXY', 'direct') if PROXY_LIST and PROXY_LIST[0] != DEFAULT_PROXY else 'direct'}")
            attempts += 1
            if attempts < max_proxy_attempts_per_video:
                current_proxy_index += 1
                print(f"  Retrying {video_id} with next proxy...")
            else:
                print(f"  All {max_proxy_attempts_per_video} proxies exhausted for {video_id}. Skipping this video.")
        finally:
            # Always unset proxy environment variables after the attempt
            if 'HTTP_PROXY' in os.environ:
                del os.environ['HTTP_PROXY']
            if 'HTTPS_PROXY' in os.environ:
                del os.environ['HTTPS_PROXY']

    if not transcript_extracted: # If after all attempts, still not extracted successfully
        # Get the original URL from metadata_df
        original_url = metadata_df[metadata_df["video_id"] == video_id]["webpage_url"].iloc[0]
        failed_video_details.append({
            "video_id": video_id,
            "webpage_url": original_url,
            "failure_reason": current_failure_reason if current_failure_reason else "Unknown failure during extraction loop"
        })
        print(f"  Final failure for {video_id}. Reason: {current_failure_reason}")

    time.sleep(DELAY_SECONDS) # Add a delay between each video extraction attempt

if all_transcripts_dfs:
    transcript_df = pd.concat(all_transcripts_dfs, ignore_index=True);
    print("\n--- Consolidated Transcript DataFrame (first 10 rows) ---")
    print(transcript_df.head(10))
else:
    print("No transcripts could be extracted for any of the provided videos.")
    transcript_df = pd.DataFrame(columns=["text", "start", "duration", "video_id"]) # Create an empty DataFrame

# Save detailed failed video information
if failed_video_details:
    failed_df = pd.DataFrame(failed_video_details)
    failed_df.to_excel("failed_transcripts.xlsx", index=False)
    print(f"\nFailed to extract transcripts for {len(failed_video_details)} videos. Details (including reasons) saved to 'failed_transcripts.xlsx'.")
else:
    print("\nAll video transcripts were extracted successfully or no videos failed this run.")

No real proxies configured. Using direct connection. IP blocking might occur.
'failed_transcripts.xlsx' found. Processing only 12 video IDs from this file.
Total video IDs to process: 12
Extracting transcript for video ID: FaL-GT3eDTk
  Attempting with direct connection (no proxy configured).
  Successfully extracted transcript for FaL-GT3eDTk.
Extracting transcript for video ID: BXj0qBJ2X7U
  Attempting with direct connection (no proxy configured).
  Failure during transcript extraction for BXj0qBJ2X7U: Transcripts Disabled: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=BXj0qBJ2X7U! This is most likely caused by:

Subtitles are disabled for this video

If you are sure that the described cause is not responsible for this error and that a transcript should be retrievable, please create an issue at https://github.com/jdepoix/youtube-transcript-api/issues. Please add which version of youtube_transcript_api you are using and provide the information needed t

# Other scraped data (list of keys/headers only)

In [ ]:
keys_list = sorted(info.keys())

# Print vertically
for key in keys_list:
    print(key)

_format_sort_fields
_has_drm
abr
acodec
age_limit
aspect_ratio
asr
audio_channels
automatic_captions
availability
average_rating
categories
channel
channel_follower_count
channel_id
channel_is_verified
channel_url
chapters
comment_count
creators
description
display_id
duration
duration_string
dynamic_range
epoch
ext
extractor
extractor_key
filesize_approx
format
format_id
format_note
formats
fps
fulltitle
heatmap
height
id
is_live
language
like_count
live_status
media_type
original_url
playable_in_embed
playlist
playlist_index
protocol
release_timestamp
release_year
requested_formats
requested_subtitles
resolution
stretched_ratio
subtitles
tags
tbr
thumbnail
thumbnails
timestamp
title
upload_date
uploader
uploader_id
uploader_url
vbr
vcodec
view_count
was_live
webpage_url
webpage_url_basename
webpage_url_domain
width


# Data at this point:

1. Scraped Metadata (see list above)
2. Transcripts (Auto-generated)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

keys_df = pd.DataFrame(keys_list, columns=["yt_dlp_keys"])
keys_df.to_excel("yt_dlp_keys.xlsx", index=False)
transcript_df.to_excel("test_transcript.xlsx", index=False)
metadata_df.to_excel("test_metadata.xlsx", index=False)

Mounted at /content/drive


In [ ]:
keys_list = sorted(info.keys())

rows = []
for key in keys_list:
    value = info.get(key)

    #Convert complex objects to string so Excel doesn't break D:
    if isinstance(value, (dict, list)):
        value = str(value)[:1000]

    rows.append({
        "key": key,
        "value": value
    })


info_df = pd.DataFrame(rows)
info_df.head(20)

info_df.to_excel("master_testdata.xlsx", index=False)